# 🛣️ ROADGUARD AI — HUẤN LUYỆN YOLOV11 MULTI-CLASS TRÊN KAGGLE
### Mô Hình Đa Lớp: Ổ Gà (0: pothole) | Vết Nứt (1: crack) | Bong Tróc (2: bong_troc)
---
### ⚙️ Hướng dẫn thiết lập Kaggle Notebook trước khi bấm Run:
1. **Accelerator:** Chọn **GPU T4 x2** (hoặc GPU P100) ở thanh bên phải (Settings -> Accelerator).
2. **Internet:** Bật **Internet ON** để tải thư viện `ultralytics`.
3. **Dataset Input:** Đảm bảo đã gắn (Add Input) Dataset chứa dữ liệu hợp nhất `roadguard-multiclass-dataset`.
4. **Môi trường làm việc:** Sử dụng `/kaggle/working` cho mọi tiến trình ghi file (vì `/kaggle/input` ở chế độ chỉ đọc).

In [ ]:
# 1. Cài đặt Ultralytics và kiểm tra thiết bị phần cứng GPU
!pip install -q --upgrade ultralytics

import os
import sys
import shutil
import torch
import yaml
from pathlib import Path

print('=' * 65)
print('🚀 KIỂM TRA PHẦN CỨNG GPU TRÊN KAGGLE:')
if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f'✓ Phát hiện {device_count} GPU khả dụng:')
    for i in range(device_count):
        print(f'   - GPU [{i}]: {torch.cuda.get_device_name(i)}')
else:
    print('⚠️ CẢNH BÁO: Không phát hiện GPU! Vui lòng bật GPU T4 x2 trong Settings.')
print('=' * 65)

In [ ]:
# 2. Tự động định vị Dataset từ /kaggle/input và sao chép sang /kaggle/working/data
input_base = Path('/kaggle/input')
working_base = Path('/kaggle/working')
target_data_dir = working_base / 'data'

# Tìm thư mục chứa dataset trong /kaggle/input
matched_dataset_dir = None
matched_weight_path = None

for p in input_base.rglob('*'):
    if p.name == 'data.yaml' and not matched_dataset_dir:
        matched_dataset_dir = p.parent
    if p.name in ['roadguard_best_15k.pt', 'best.pt'] and not matched_weight_path:
        matched_weight_path = p

if not matched_dataset_dir:
    raise FileNotFoundError(
        '❌ Không tìm thấy data.yaml trong /kaggle/input! '
        'Vui lòng bấm "+ Add Input" ở góc trên bên phải để thêm Dataset vào Notebook.'
    )

print(f'✓ Tìm thấy Dataset nguồn tại: {matched_dataset_dir}')
if matched_weight_path:
    print(f'✓ Tìm thấy Trọng số nền tảng tại: {matched_weight_path}')
else:
    print('ℹ️ Không thấy file roadguard_best_15k.pt, sẽ sử dụng yolo11n.pt làm trọng số khởi đầu.')

# Sao chép sang /kaggle/working/data để có toàn quyền ghi file và tạo labels.cache
if target_data_dir.exists():
    shutil.rmtree(target_data_dir)
target_data_dir.mkdir(parents=True, exist_ok=True)

print('📦 Đang sao chép dataset sang /kaggle/working/data/ (vui lòng chờ)...')
for item in ['train', 'valid', 'test']:
    src_sub = matched_dataset_dir / item
    if src_sub.exists():
        shutil.copytree(src_sub, target_data_dir / item)
        print(f'   ✓ Đã sao chép phân tập {item}: {len(list((target_data_dir / item / "images").glob("*.*")))} ảnh')

# Cập nhật data.yaml với đường dẫn tuyệt đối chuẩn /kaggle/working/data
with open(matched_dataset_dir / 'data.yaml', 'r', encoding='utf-8') as f:
    yaml_cfg = yaml.safe_load(f)

yaml_cfg['path'] = str(target_data_dir.resolve())
yaml_cfg['train'] = 'train/images'
yaml_cfg['val'] = 'valid/images'
yaml_cfg['test'] = 'test/images'
yaml_cfg['nc'] = 3
yaml_cfg['names'] = {0: 'pothole', 1: 'crack', 2: 'bong_troc'}

working_yaml = target_data_dir / 'data.yaml'
with open(working_yaml, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_cfg, f, default_flow_style=False, sort_keys=False)

# Sao chép trọng số nền tảng
local_weight = working_base / 'roadguard_best_15k.pt'
if matched_weight_path and matched_weight_path.exists():
    shutil.copy2(matched_weight_path, local_weight)
else:
    local_weight = 'yolo11n.pt'

print(f'📄 File cấu hình huấn luyện đã sẵn sàng: {working_yaml}')
print(f'🎯 File trọng số ban đầu: {local_weight}')

## 🧪 Bước 2a: Smoke Test (5 Epochs)
> Chạy kiểm thử nhanh 5 epochs để xác nhận đường truyền dữ liệu (data loader), nạp nhãn đa lớp và tài nguyên GPU hoạt động trơn tru không phát sinh lỗi.

In [ ]:
from ultralytics import YOLO

print('=' * 65)
print('🚀 TIẾN HÀNH SMOKE TEST (5 EPOCHS)...')
print('=' * 65)

smoke_model = YOLO(str(local_weight))
smoke_results = smoke_model.train(
    data=str(working_yaml),
    epochs=5,
    imgsz=640,
    batch=16,
    project='/kaggle/working/runs/detect',
    name='smoke_test',
    exist_ok=True,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu'
)
print('🎉 SMOKE TEST 5 EPOCHS THÀNH CÔNG! SẴN SÀNG HUẤN LUYỆN TOÀN DIỆN 120 EPOCHS.')

## 🏋️ Bước 2b: Huấn Luyện Toàn Diện (Full Training 120 Epochs)
> Huấn luyện toàn diện 120 epochs với `patience=20`, `batch=16`, `imgsz=640`, lưu checkpoint tốt nhất tại `/kaggle/working/runs/detect/roadguard_multiclass_v1/weights/best.pt`.

In [ ]:
print('=' * 65)
print('🚀 BẮT ĐẦU HUẤN LUYỆN FULL ROADGUARD MULTICLASS V1 (120 EPOCHS)...')
print('=' * 65)

# Khởi tạo lại từ trọng số nền tảng tốt nhất ban đầu
full_model = YOLO(str(local_weight))

full_results = full_model.train(
    data=str(working_yaml),
    epochs=120,
    patience=20,
    batch=16,
    imgsz=640,
    name='roadguard_multiclass_v1',
    project='/kaggle/working/runs/detect',
    exist_ok=True,
    save=True,
    save_period=10,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu'
)
print('🎉 HUẤN LUYỆN TOÀN DIỆN 120 EPOCHS ĐÃ HOÀN TẤT THÀNH CÔNG!')

## 📊 Bước 2c: Đánh Giá Chi Tiết Per-Class AP, Confusion Matrix & Nghiệm Thu (Ngưỡng AP $\ge$ 0.6)
> Đánh giá mô hình tốt nhất trên tập Test độc lập, kiểm tra ngưỡng đạt chuẩn cho từng lớp và hiển thị ma trận nhầm lẫn.

In [ ]:
from IPython.display import Image, display

run_dir = Path('/kaggle/working/runs/detect/roadguard_multiclass_v1')
best_pt = run_dir / 'weights' / 'best.pt'

if not best_pt.exists():
    # Fallback nếu dùng tên khác
    best_pt = list(Path('/kaggle/working/runs/detect').rglob('best.pt'))[0]

print(f'🏆 Nạp trọng số tốt nhất để đánh giá: {best_pt}')
trained_model = YOLO(str(best_pt))

# Chạy validation trên tập Test, thiết lập project/name tường minh không gây lồng thư mục
metrics = trained_model.val(
    data=str(working_yaml),
    split='test',
    project='/kaggle/working/runs/detect',
    name='roadguard_multiclass_v1_eval',
    exist_ok=True,
    plots=True
)
eval_dir = Path(metrics.save_dir)

print('=' * 75)
print('📋 BẢNG ĐÁNH GIÁ CHI TIẾT TỪNG LỚP KHUYẾT TẬT (PER-CLASS AP METRICS):')
print('=' * 75)

passed_classes = []
failed_classes = []

for i, name in trained_model.names.items():
    ap50 = metrics.box.ap50[i] * 100 if i < len(metrics.box.ap50) else 0.0
    ap50_95 = metrics.box.ap[i] * 100 if i < len(metrics.box.ap) else 0.0
    status = '✅ ĐẠT CHUẨN' if ap50 >= 60.0 else '⚠️ DƯỚI NGƯỠNG'
    print(f' - Lớp [{i}] {name:12s} | AP@50 = {ap50:6.2f}% | AP@50-95 = {ap50_95:6.2f}% | Trạng thái: {status}')
    if ap50 >= 60.0:
        passed_classes.append((i, name, ap50))
    else:
        failed_classes.append((i, name, ap50))

print('-' * 75)
print(f'🎯 ĐỘ CHÍNH XÁC TOÀN DIỆN: mAP@50 = {metrics.box.map50 * 100:.2f}% | mAP@50-95 = {metrics.box.map * 100:.2f}%')
print('=' * 75)

if failed_classes:
    print('⚠️ DANH SÁCH CÁC LỚP CÓ ĐỘ CHÍNH XÁC AP < 0.6 CẦN BỔ SUNG DỮ LIỆU THỰC ĐỊA:')
    for cid, cname, cap in failed_classes:
        print(f'   👉 Lớp [{cid}] {cname}: AP@50 = {cap:.2f}% (< 60.0%) -> Kế hoạch: Bổ sung mẫu từ Flycam qua KDV Inspector.')

# Hiển thị ma trận nhầm lẫn Confusion Matrix
print('\n🖼️ MA TRẬN NHẦM LẪN (CONFUSION MATRIX):')
cm_file = eval_dir / 'confusion_matrix.png'
if cm_file.exists():
    display(Image(str(cm_file)))
elif (run_dir / 'confusion_matrix.png').exists():
    display(Image(str(run_dir / 'confusion_matrix.png')))

# Xuất file trọng số ra thư mục gốc /kaggle/working để người dùng tải về dễ dàng
export_weight = working_base / 'roadguard_multiclass_v1_best.pt'
shutil.copy2(best_pt, export_weight)
print(f'\n💾 Đã xuất file trọng số hoàn chỉnh tại: {export_weight}')
print('👉 Bạn có thể tải file này trực tiếp từ tab Output bên phải màn hình Kaggle!')